In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
url = "https://www.ccny.cuny.edu/registrar/fall"

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)

Status code: 403


In [15]:
with open("fall2021.html", "r", encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")

print(soup.title)

<title>Fall 2021 Academic Calendar | The City College of New York</title>


In [16]:
tables = soup.find_all("table")

print("Number of tables:", len(tables))

Number of tables: 7


In [17]:
for i, table in enumerate(tables):
    print("TABLE", i)
    print(table.get_text(" | ", strip=True)[:500])
    print()

TABLE 0
Search Query | × | search

TABLE 1
Search Query | ×

TABLE 2
× | search

TABLE 3
×

TABLE 4
DATES | DAYS | August 01 | Sunday | Application for degree for January and February 2022 begins | August 18 | Wednesday | Last day to apply for Study Abroad | August 24 | Tuesday | Last day of Registration; | Last day to file ePermit for the Fall 2021; | Last day to drop classes for 100% tuition refund; | August 25 | Wednesday | Start of Fall Term; | Classes begin; | Initial Registration Appeals begin; | August 25 - 31 | Wednesday - Tuesday | Change of program period; late fees apply | August 26

TABLE 5


TABLE 6




In [21]:
table = tables[4]

rows = table.find_all("tr")

print("Number of rows:", len(rows))

Number of rows: 37


In [22]:
for row in rows[:5]:
    print(row.get_text(" | ", strip=True))

DATES | DAYS
August 01 | Sunday | Application for degree for January and February 2022 begins
August 18 | Wednesday | Last day to apply for Study Abroad
August 24 | Tuesday | Last day of Registration; | Last day to file ePermit for the Fall 2021; | Last day to drop classes for 100% tuition refund;
August 25 | Wednesday | Start of Fall Term; | Classes begin; | Initial Registration Appeals begin;


In [23]:
data = []

for row in rows:
    cells = row.find_all(["td", "th"])

    if len(cells) >= 3:
        date = cells[0].get_text(" ", strip=True)
        dow = cells[1].get_text(" ", strip=True)
        text = cells[2].get_text(" ", strip=True)

        data.append([date, dow, text])

In [24]:
data[:10]

[['DATES', 'DAYS', ''],
 ['August 01',
  'Sunday',
  'Application for degree for January and February 2022 begins'],
 ['August 18', 'Wednesday', 'Last day to apply for Study Abroad'],
 ['August 24',
  'Tuesday',
  'Last day of Registration; Last day to file ePermit for the Fall 2021; Last day to drop classes for 100% tuition refund;'],
 ['August 25',
  'Wednesday',
  'Start of Fall Term; Classes begin; Initial Registration Appeals begin;'],
 ['August 25 - 31',
  'Wednesday - Tuesday',
  'Change of program period; late fees apply'],
 ['August 26', 'Thursday', 'Last day for Independent Study'],
 ['August 28', 'Saturday', 'First day of Saturday Classes'],
 ['August 31',
  'Tuesday',
  "Last day to add a class to an existing enrollment; Last day for 75% tuition refund; Financial Aid Certification Enrollment Status date; Last day to apply for Audit option; Last day to drop without the grade of 'WD'; Initial Registration Appeals end;"],
 ['September 01',
  'Wednesday',
  "Verification of Enr

In [ ]:
df = pd.DataFrame(data, columns=["date", "dow", "text"])

df.head()

,date,dow,text
0,DATES,DAYS,
1,August 01,Sunday,Application for degree for January and Februar...
2,August 18,Wednesday,Last day to apply for Study Abroad
3,August 24,Tuesday,Last day of Registration; Last day to file ePe...
4,August 25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...


In [26]:
df = df.iloc[1:].reset_index(drop=True)

df.head()

,date,dow,text
0,August 01,Sunday,Application for degree for January and Februar...
1,August 18,Wednesday,Last day to apply for Study Abroad
2,August 24,Tuesday,Last day of Registration; Last day to file ePe...
3,August 25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
4,August 25 - 31,Wednesday - Tuesday,Change of program period; late fees apply


In [27]:
def clean_date(date_string):
    date_string = date_string.strip()

    if " - " in date_string:
        date_string = date_string.split(" - ")[0].strip()

    if date_string.startswith("January"):
        year = 2022
    else:
        year = 2021

    return pd.to_datetime(f"{date_string} {year}")

In [28]:
df["date"] = df["date"].apply(clean_date)

df.head(10)

,date,dow,text
0,2021-08-01,Sunday,Application for degree for January and Februar...
1,2021-08-18,Wednesday,Last day to apply for Study Abroad
2,2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
3,2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
4,2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply
5,2021-08-26,Thursday,Last day for Independent Study
6,2021-08-28,Saturday,First day of Saturday Classes
7,2021-08-31,Tuesday,Last day to add a class to an existing enrollm...
8,2021-09-01,Wednesday,Verification of Enrollment rosters available t...
9,2021-09-03,Friday - Wednesday,No classes scheduled


In [29]:
df["date"] = df["date"].dt.date

df.head()

df = df.set_index("date")

df.head()

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply


In [31]:
df

,dow,text
date,,
2021-08-01,Sunday,Application for degree for January and Februar...
2021-08-18,Wednesday,Last day to apply for Study Abroad
2021-08-24,Tuesday,Last day of Registration; Last day to file ePe...
2021-08-25,Wednesday,Start of Fall Term; Classes begin; Initial Reg...
2021-08-25,Wednesday - Tuesday,Change of program period; late fees apply
2021-08-26,Thursday,Last day for Independent Study
2021-08-28,Saturday,First day of Saturday Classes
2021-08-31,Tuesday,Last day to add a class to an existing enrollm...
2021-09-01,Wednesday,Verification of Enrollment rosters available t...


The CCNY Calendar link returned a HTTP 403 Forbidden when I tried to access it directly through python requests probably because of better security from CCNY's end.
So I saved the webpage locally as HTML and went through it with BeautifulSoup.